# 📖 Notebook 3: Route Matching & Segment Leaderboards

Strava's killer feature: **segments**. A segment is a specific stretch of road
that athletes race on. When your GPS trace passes through a segment, Strava
automatically records your time and ranks you on a leaderboard.

In this notebook we build the segment system end-to-end.

## Learning Objectives

By the end of this notebook, you'll understand:
- What segments are and how they're defined
- How to detect when a GPS route crosses a segment (**route matching**)
- How **Redis Sorted Sets** power O(log N) leaderboards
- How to filter leaderboards by city, country, and time range
- The scaling trade-offs of different leaderboard approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/strava
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `strava_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import math
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "strava_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def haversine(lat1, lon1, lat2, lon2):
    """Distance in metres between two GPS points."""
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🏁 What Is a Segment?

A **segment** is a stretch of road defined by a start point and an end point.
Think of it as a race course — everyone who passes through it gets timed.

Examples:
- *Embarcadero Sprint* — 850 m along the SF waterfront
- *Golden Gate Park Climb* — 2.8 km through the park
- *Central Park North Loop* — 1.2 km in NYC

A **segment effort** is one athlete's attempt at a segment.
The fastest effort wins the **leaderboard** (King/Queen of the Mountain).

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# List all segments
cur.execute("SELECT id, name, type, distance_m, city FROM segments ORDER BY id")
segments = cur.fetchall()

print("Available Segments:")
print(f"{'ID':>4} {'Name':<30} {'Type':<6} {'Distance':>10} {'City':<15}")
print("-" * 70)
for s in segments:
    print(f"{s['id']:>4} {s['name']:<30} {s['type']:<6} {s['distance_m']:>8.0f} m  {s['city']:<15}")

# Show existing efforts for the Embarcadero Sprint
print()
cur.execute("""
    SELECT se.id, u.username, se.elapsed_s, s.name
    FROM segment_efforts se
    JOIN users u ON se.user_id = u.id
    JOIN segments s ON se.segment_id = s.id
    WHERE se.segment_id = 1
    ORDER BY se.elapsed_s
    LIMIT 5;
""")
efforts = cur.fetchall()

print(f"Embarcadero Sprint — Top efforts:")
for i, e in enumerate(efforts, 1):
    mins = e['elapsed_s'] // 60
    secs = e['elapsed_s'] % 60
    print(f"  #{i} {e['username']:<10} {mins}:{secs:02d}")

conn.close()

## 🔍 Route Matching: Did You Cross a Segment?

When an activity is uploaded, we need to check if the GPS trace passes through
any known segments. This is **route matching**.

### Simple approach (good enough for interviews):

1. For each segment, check if any route point is **near the start** AND any later point is **near the end**
2. "Near" means within a threshold distance (e.g., 50 metres)
3. If both match, calculate the time between the start-match and end-match points

```
Segment: Embarcadero Sprint
  Start: (37.7955, -122.3935)
  End:   (37.7899, -122.3860)

Your GPS trace:
  Point 1: (37.7954, -122.3936) ← within 50 m of segment start ✅
  Point 2: (37.7948, -122.3925)
  ...
  Point 9: (37.7900, -122.3861) ← within 50 m of segment end ✅

→ Matched! Effort time = timestamp(point 9) - timestamp(point 1)
```

In [ ]:
MATCH_THRESHOLD_M = 100  # how close a GPS point must be to a segment endpoint

def match_route_to_segments(activity_id):
    """
    Check if an activity's GPS trace passes through any known segments.
    Returns a list of matched segments with elapsed time.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get the activity's route points in order
    cur.execute("""
        SELECT seq, latitude, longitude, recorded_at
        FROM route_points
        WHERE activity_id = %s
        ORDER BY seq;
    """, (activity_id,))
    points = cur.fetchall()

    # Get the activity type to only match same-type segments
    cur.execute("SELECT type, user_id FROM activities WHERE id = %s", (activity_id,))
    activity = cur.fetchone()

    # Get all segments of the same type
    cur.execute("SELECT * FROM segments WHERE type = %s", (activity['type'],))
    segments = cur.fetchall()
    conn.close()

    matches = []

    for seg in segments:
        start_match = None
        end_match = None

        for pt in points:
            # Check distance to segment start
            dist_to_start = haversine(
                pt['latitude'], pt['longitude'],
                seg['start_lat'], seg['start_lon']
            )
            if dist_to_start <= MATCH_THRESHOLD_M and start_match is None:
                start_match = pt

            # Check distance to segment end (only after we matched the start)
            if start_match:
                dist_to_end = haversine(
                    pt['latitude'], pt['longitude'],
                    seg['end_lat'], seg['end_lon']
                )
                if dist_to_end <= MATCH_THRESHOLD_M:
                    end_match = pt
                    break  # done — found start and end

        if start_match and end_match:
            elapsed = (end_match['recorded_at'] - start_match['recorded_at']).total_seconds()
            matches.append({
                'segment_id': seg['id'],
                'segment_name': seg['name'],
                'elapsed_s': int(elapsed),
                'start_seq': start_match['seq'],
                'end_seq': end_match['seq'],
                'user_id': activity['user_id'],
            })

    return matches

# Match Alice's Embarcadero run (activity 1)
matches = match_route_to_segments(activity_id=1)

print(f"Route matching for Activity #1 (Alice's Embarcadero Run):")
print()
if matches:
    for m in matches:
        mins = m['elapsed_s'] // 60
        secs = m['elapsed_s'] % 60
        print(f"  ✅ Matched: {m['segment_name']}")
        print(f"     Points {m['start_seq']} → {m['end_seq']}")
        print(f"     Effort time: {mins}:{secs:02d}")
else:
    print("  No segments matched.")

print()
print("💡 In production, this matching runs automatically when activities are uploaded.")
print("   PostGIS ST_DWithin makes the proximity check efficient even with millions of points.")

In [ ]:
# PostGIS version: same matching but using spatial SQL
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    -- For each segment, find route points near the start and end
    WITH start_matches AS (
        SELECT s.id AS segment_id, s.name, rp.seq, rp.recorded_at,
               ST_Distance(rp.geom, ST_Point(s.start_lon, s.start_lat)::geography) AS dist
        FROM segments s
        CROSS JOIN route_points rp
        WHERE rp.activity_id = 1
          AND s.type = 'RUN'
          AND ST_DWithin(rp.geom, ST_Point(s.start_lon, s.start_lat)::geography, 100)
    ),
    end_matches AS (
        SELECT s.id AS segment_id, rp.seq, rp.recorded_at,
               ST_Distance(rp.geom, ST_Point(s.end_lon, s.end_lat)::geography) AS dist
        FROM segments s
        CROSS JOIN route_points rp
        WHERE rp.activity_id = 1
          AND s.type = 'RUN'
          AND ST_DWithin(rp.geom, ST_Point(s.end_lon, s.end_lat)::geography, 100)
    )
    SELECT sm.segment_id, sm.name,
           sm.seq AS start_seq, em.seq AS end_seq,
           EXTRACT(EPOCH FROM (em.recorded_at - sm.recorded_at))::int AS elapsed_s
    FROM start_matches sm
    JOIN end_matches em ON sm.segment_id = em.segment_id
    WHERE em.seq > sm.seq;
""")
pg_matches = cur.fetchall()
conn.close()

print("PostGIS route matching (pure SQL):")
for m in pg_matches:
    mins = m['elapsed_s'] // 60
    secs = m['elapsed_s'] % 60
    print(f"  ✅ {m['name']}: points {m['start_seq']}→{m['end_seq']}, time {mins}:{secs:02d}")

print()
print("💡 PostGIS uses spatial indexes (GIST) — this scales to millions of route points!")

## 🏆 Leaderboards with Redis Sorted Sets

A **Sorted Set** in Redis is a collection where each member has a **score**.
Members are automatically sorted by score, and you can query ranges efficiently.

Perfect for leaderboards!

```
Key:    leaderboard:segment:1         (Embarcadero Sprint)
Member: user_id                       (who)
Score:  best elapsed time in seconds  (lower is better)
```

### Key Operations (all O(log N)):

| Operation | Redis Command | What It Does |
|-----------|--------------|---------------|
| Add/update | `ZADD` | Add a score or update if lower |
| Top N | `ZRANGE` | Get the N best scores |
| My rank | `ZRANK` | Find where I am in the ranking |
| My score | `ZSCORE` | Get my best time |

In [ ]:
r = get_redis()

def build_leaderboard_from_db(segment_id):
    """
    Build a Redis leaderboard from existing segment efforts in Postgres.
    For each user, only their best (lowest) time counts.
    """
    conn = get_db()
    cur = conn.cursor()

    # Get best effort per user for this segment
    cur.execute("""
        SELECT user_id, MIN(elapsed_s) AS best_time
        FROM segment_efforts
        WHERE segment_id = %s
        GROUP BY user_id;
    """, (segment_id,))

    key = f"leaderboard:segment:{segment_id}"
    r.delete(key)

    count = 0
    for user_id, best_time in cur.fetchall():
        # Score = elapsed_s (lower is better, so lower score = higher rank)
        r.zadd(key, {str(user_id): best_time})
        count += 1

    conn.close()
    return count

# Build leaderboards for all segments
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, name FROM segments")
all_segments = cur.fetchall()
conn.close()

for seg_id, seg_name in all_segments:
    n = build_leaderboard_from_db(seg_id)
    print(f"  Built leaderboard for '{seg_name}': {n} athletes")

print()
print("💡 Open RedisInsight to see the sorted sets!")

In [ ]:
def get_leaderboard(segment_id, top_n=10):
    """Get the top N athletes for a segment from Redis."""
    key = f"leaderboard:segment:{segment_id}"
    # ZRANGE with WITHSCORES returns [(member, score), ...]
    # Lower score = faster time = better rank
    results = r.zrange(key, 0, top_n - 1, withscores=True)
    
    # Look up usernames
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    leaderboard = []
    for user_id_str, score in results:
        cur.execute("SELECT username, display_name, city FROM users WHERE id = %s",
                    (int(user_id_str),))
        user = cur.fetchone()
        elapsed = int(score)
        leaderboard.append({
            'user_id': int(user_id_str),
            'username': user['username'],
            'display_name': user['display_name'],
            'city': user['city'],
            'elapsed_s': elapsed,
            'time_str': f"{elapsed // 60}:{elapsed % 60:02d}",
        })
    conn.close()
    return leaderboard

# Show leaderboard for each segment
for seg_id, seg_name in all_segments:
    lb = get_leaderboard(seg_id, top_n=5)
    print(f"🏆 {seg_name} — Top 5:")
    for i, entry in enumerate(lb, 1):
        crown = '👑' if i == 1 else '  '
        print(f"  {crown} #{i} {entry['display_name']:<20} {entry['time_str']:>6} "
              f"  ({entry['city']})")
    print()

In [ ]:
# Demonstrate real-time leaderboard updates
def record_segment_effort(segment_id, user_id, elapsed_s):
    """
    When an athlete completes a segment, update both Postgres and Redis.
    Only updates the leaderboard if this is a personal best.
    """
    key = f"leaderboard:segment:{segment_id}"

    # Check current best in Redis
    current_best = r.zscore(key, str(user_id))

    if current_best is None or elapsed_s < current_best:
        # New personal best! Update Redis
        r.zadd(key, {str(user_id): elapsed_s})
        is_pb = True
    else:
        is_pb = False

    # Get their new rank
    rank = r.zrank(key, str(user_id))

    return {
        'personal_best': is_pb,
        'rank': rank + 1 if rank is not None else None,
        'time': f"{elapsed_s // 60}:{elapsed_s % 60:02d}",
        'previous_best': f"{int(current_best) // 60}:{int(current_best) % 60:02d}" if current_best else 'N/A'
    }

# Carol (user 3) just smashed the Embarcadero Sprint!
result = record_segment_effort(segment_id=1, user_id=3, elapsed_s=195)
print("Carol just finished the Embarcadero Sprint in 3:15!")
print(f"  Personal best: {'YES 🎉' if result['personal_best'] else 'No'}")
print(f"  Previous best: {result['previous_best']}")
print(f"  Current rank:  #{result['rank']}")
print()

# Show updated leaderboard
lb = get_leaderboard(segment_id=1, top_n=5)
print("Updated Embarcadero Sprint leaderboard:")
for i, entry in enumerate(lb, 1):
    crown = '👑' if i == 1 else '  '
    new = ' ← NEW!' if entry['username'] == 'carol' else ''
    print(f"  {crown} #{i} {entry['display_name']:<20} {entry['time_str']:>6}{new}")
print()
print("💡 The leaderboard updated instantly — no database query needed!")
print("   ZADD + ZRANGE are both O(log N) in Redis.")

## 🌍 Filtered Leaderboards: By City and Country

Athletes want to see how they rank *locally*, not just globally.

**Solution**: maintain separate sorted sets per filter:

```
leaderboard:segment:1              ← global
leaderboard:segment:1:country:USA  ← USA only
leaderboard:segment:1:city:SF      ← San Francisco only
```

When an effort is recorded, we update ALL relevant sorted sets.

In [ ]:
def build_filtered_leaderboards(segment_id):
    """
    Build global, country, and city leaderboards for a segment.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT se.user_id, u.city, u.country, MIN(se.elapsed_s) AS best_time
        FROM segment_efforts se
        JOIN users u ON se.user_id = u.id
        WHERE se.segment_id = %s
        GROUP BY se.user_id, u.city, u.country;
    """, (segment_id,))

    # Clear existing keys
    for key in r.keys(f"leaderboard:segment:{segment_id}:*"):
        r.delete(key)

    counts = {'global': 0}
    for row in cur.fetchall():
        uid = str(row['user_id'])
        bt = row['best_time']

        # Global leaderboard (already built, but let's rebuild)
        r.zadd(f"leaderboard:segment:{segment_id}", {uid: bt})
        counts['global'] += 1

        # Country leaderboard
        country_key = f"leaderboard:segment:{segment_id}:country:{row['country']}"
        r.zadd(country_key, {uid: bt})
        counts[row['country']] = counts.get(row['country'], 0) + 1

        # City leaderboard
        city_key = f"leaderboard:segment:{segment_id}:city:{row['city']}"
        r.zadd(city_key, {uid: bt})

    conn.close()
    return counts

# Build filtered leaderboards for Embarcadero Sprint
counts = build_filtered_leaderboards(segment_id=1)
print("Built filtered leaderboards for Embarcadero Sprint:")
for k, v in counts.items():
    print(f"  {k}: {v} athletes")
print()

# Show global vs USA vs SF leaderboard
for label, key_suffix in [('🌍 Global', ''), ('🇺🇸 USA', ':country:USA'), ('🏙️ San Francisco', ':city:San Francisco')]:
    key = f"leaderboard:segment:1{key_suffix}"
    top = r.zrange(key, 0, 2, withscores=True)
    print(f"{label} top 3:")
    conn = get_db()
    cur = conn.cursor()
    for i, (uid, score) in enumerate(top, 1):
        cur.execute("SELECT username FROM users WHERE id = %s", (int(uid),))
        username = cur.fetchone()[0]
        t = int(score)
        print(f"  #{i} {username:<10} {t//60}:{t%60:02d}")
    conn.close()
    print()

## 📊 Distance Leaderboards with ZINCRBY

Another type of leaderboard: **total distance** across all activities.

When a user completes an activity, we increment their total distance:

```python
redis.zincrby("leaderboard:distance:run:global", distance_m, user_id)
```

`ZINCRBY` atomically adds to the score — perfect for cumulative stats.

In [ ]:
def build_distance_leaderboard():
    """Build a total-distance leaderboard from all completed activities."""
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""
        SELECT a.user_id, a.type, u.country,
               SUM(a.distance_m) AS total_distance
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
        GROUP BY a.user_id, a.type, u.country;
    """)

    for key in r.keys("leaderboard:distance:*"):
        r.delete(key)

    for user_id, activity_type, country, total_dist in cur.fetchall():
        uid = str(user_id)
        # Global by type
        r.zadd(f"leaderboard:distance:{activity_type.lower()}:global", {uid: float(total_dist)})
        # Country by type
        r.zadd(f"leaderboard:distance:{activity_type.lower()}:{country}", {uid: float(total_dist)})

    conn.close()

build_distance_leaderboard()

# Show top runners by total distance
print("🏃 Top Runners by Total Distance (Global):")
top_runners = r.zrevrange("leaderboard:distance:run:global", 0, 9, withscores=True)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
for i, (uid, dist) in enumerate(top_runners, 1):
    cur.execute("SELECT username, display_name, city FROM users WHERE id = %s", (int(uid),))
    u = cur.fetchone()
    print(f"  #{i:>2} {u['display_name']:<20} {dist/1000:>8.1f} km  ({u['city']})")

print()
print("🚴 Top Cyclists by Total Distance (Global):")
top_riders = r.zrevrange("leaderboard:distance:ride:global", 0, 9, withscores=True)
for i, (uid, dist) in enumerate(top_riders, 1):
    cur.execute("SELECT username, display_name, city FROM users WHERE id = %s", (int(uid),))
    u = cur.fetchone()
    print(f"  #{i:>2} {u['display_name']:<20} {dist/1000:>8.1f} km  ({u['city']})")

conn.close()

In [ ]:
# Demonstrate real-time increment with ZINCRBY
print("Alice just completed a 10 km run!")
print()

# Before
before = r.zscore("leaderboard:distance:run:global", "1")
print(f"  Before: {before/1000:.1f} km total")

# ZINCRBY atomically adds to the score
r.zincrby("leaderboard:distance:run:global", 10000, "1")  # 10 km in metres
r.zincrby("leaderboard:distance:run:USA", 10000, "1")

# After
after = r.zscore("leaderboard:distance:run:global", "1")
rank = r.zrevrank("leaderboard:distance:run:global", "1")
print(f"  After:  {after/1000:.1f} km total")
print(f"  Rank:   #{rank + 1}")
print()
print("💡 ZINCRBY is atomic — safe even with millions of concurrent updates!")
print("   No need to read-modify-write. Redis handles it.")

## 📅 Time-Range Leaderboards: Weekly King/Queen of the Mountain

Strava shows leaderboards for **this week**, **this month**, **this year**, and **all time**.

How? A separate sorted set per time bucket:

```
leaderboard:segment:1                       ← all time
leaderboard:segment:1:week:2026-W16         ← ISO week 16 of 2026
leaderboard:segment:1:month:2026-04         ← April 2026
```

When an effort is recorded, we update *every* relevant bucket with a single pipeline.
Old weekly keys expire automatically via Redis TTL — no clean-up job needed.

**Bad**: `SELECT ... FROM segment_efforts WHERE started_at BETWEEN ...`
scans millions of rows on every page view.  
**Best**: pre-aggregate into per-bucket sorted sets; reads are O(log N).

In [ ]:
from datetime import datetime, timedelta

def week_bucket(ts: datetime) -> str:
    """ISO week key like '2026-W16'."""
    y, w, _ = ts.isocalendar()
    return f"{y}-W{w:02d}"

def record_effort_time_bucketed(segment_id, user_id, elapsed_s, when: datetime):
    """Update all-time, weekly, and monthly leaderboards for one segment effort."""
    pipe = r.pipeline()
    # All-time (lowest score = fastest)
    pipe.zadd(f"leaderboard:segment:{segment_id}",
              {str(user_id): elapsed_s}, lt=True)
    # Weekly bucket — expires 90 days after the week ends
    wkey = f"leaderboard:segment:{segment_id}:week:{week_bucket(when)}"
    pipe.zadd(wkey, {str(user_id): elapsed_s}, lt=True)
    pipe.expire(wkey, 90 * 24 * 3600)
    # Monthly bucket — expires 400 days later
    mkey = f"leaderboard:segment:{segment_id}:month:{when:%Y-%m}"
    pipe.zadd(mkey, {str(user_id): elapsed_s}, lt=True)
    pipe.expire(mkey, 400 * 24 * 3600)
    pipe.execute()

# Replay every segment effort from Postgres into time-bucketed sets
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT segment_id, user_id, elapsed_s, started_at
    FROM segment_efforts
    ORDER BY started_at;
""")
n = 0
for segment_id, user_id, elapsed_s, started_at in cur.fetchall():
    record_effort_time_bucketed(segment_id, user_id, elapsed_s, started_at)
    n += 1
conn.close()
print(f"Replayed {n} efforts into time-bucketed leaderboards.")

# Show this week's top 3 on the Embarcadero Sprint
this_week = week_bucket(datetime.now())
wkey = f"leaderboard:segment:1:week:{this_week}"
top = r.zrange(wkey, 0, 2, withscores=True)

print()
print(f"🏆 Embarcadero Sprint — Week {this_week} top 3:")
if not top:
    print("  (no efforts this week)")
else:
    conn = get_db(); cur = conn.cursor()
    for i, (uid, score) in enumerate(top, 1):
        cur.execute("SELECT username FROM users WHERE id = %s", (int(uid),))
        username = cur.fetchone()[0]
        t = int(score)
        print(f"  #{i} {username:<10} {t//60}:{t%60:02d}")
    conn.close()

# Key design choice: TTL on weekly keys keeps storage bounded automatically
ttl = r.ttl(wkey)
print()
print(f"💡 TTL on '{wkey}' = {ttl} seconds (~{ttl//86400} days).")
print("   Redis auto-deletes old weekly leaderboards — no cron job needed.")


## 📈 Scaling Leaderboards: Three Approaches

The Hello Interview breakdown covers three approaches. Let's compare them.

| Approach | Latency | Freshness | Complexity | When to Use |
|----------|---------|-----------|------------|-------------|
| **Naive SQL** | Slow (full table scan) | Real-time | Simple | Prototyping only |
| **Periodic aggregation** | Fast (pre-computed) | Stale (minutes/hours) | Medium | Daily/weekly leaderboards |
| **Redis Sorted Sets** | Fast (O(log N)) | Real-time | Medium | Segment & live leaderboards |

In [ ]:
# Approach 1: Naive SQL (slow at scale)
conn = get_db()
cur = conn.cursor()

start = time.time()
for _ in range(50):
    cur.execute("""
        SELECT u.username, SUM(a.distance_m) AS total_distance
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE' AND a.type = 'RUN'
        GROUP BY u.username
        ORDER BY total_distance DESC
        LIMIT 10;
    """)
    cur.fetchall()
sql_avg = ((time.time() - start) / 50) * 1000
conn.close()

# Approach 3: Redis Sorted Set (fast)
start = time.time()
for _ in range(50):
    r.zrevrange("leaderboard:distance:run:global", 0, 9, withscores=True)
redis_avg = ((time.time() - start) / 50) * 1000

print("Leaderboard query latency (50 queries each):")
print(f"  Naive SQL (GROUP BY + ORDER BY): {sql_avg:.2f} ms")
print(f"  Redis Sorted Set (ZREVRANGE):    {redis_avg:.2f} ms")
print(f"  Speedup:                         {sql_avg/redis_avg:.1f}×")
print()
print("💡 With our tiny dataset, SQL is already slower.")
print("   At 36B+ activities, the SQL approach is completely impractical.")
print("   Redis Sorted Sets scale to millions of members with sub-ms latency.")

## 🧹 Cleanup

In [ ]:
# Clean up all Redis keys
r = get_redis()
for pattern in ['leaderboard:*']:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)
        print(f"🧹 Deleted {len(keys)} keys matching '{pattern}'")
print("🧹 Done!")

## 📚 Summary

### Key Takeaways

1. **Segments** are defined by start/end GPS points — athletes race through them
2. **Route matching** checks if a GPS trace passes near segment endpoints
3. **PostGIS ST_DWithin** makes proximity checks fast with spatial indexes
4. **Redis Sorted Sets** are the perfect data structure for leaderboards:
   - `ZADD` — add or update a score
   - `ZRANGE` / `ZREVRANGE` — get top/bottom N
   - `ZRANK` — find a user's position
   - `ZINCRBY` — atomically increment a score
5. **Filtered leaderboards** (by city/country) use separate sorted sets
6. At scale, Redis sorted sets are orders of magnitude faster than SQL GROUP BY

### System Design Interview Tips

- Start with the simple SQL approach, then explain why it won't scale
- Propose Redis Sorted Sets as the scalable solution
- Mention filtered leaderboards (separate keys per country/city)
- For time-range filtering, combine sorted sets with hashes
- Always discuss the consistency trade-off between Redis and the primary database